In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from mpl_toolkits.axes_grid1 import inset_locator
import matplotlib as mpl

# ==========================================
# 1. PRL FORMATTING SETTINGS
# ==========================================
mpl.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 18
})

# ==========================================
# 2. LOAD DATA FROM SUPERCOMPUTER
# ==========================================
filename = "/Users/suttamchanda/Downloads/PRL_Dataset_N1000_Grid10.npz"  # Change to your actual filename
print(f"Loading HPC Data from {filename}...")
data = np.load(filename)

pos_async = data['pos_async']
dt_grid_snap = data['dt_grid_snap']
e_hist_async = data['e_hist_async']
e_hist_glob = data['e_hist_glob']
dt_hist_async = data['dt_hist_async']
dt_hist_glob = data['dt_hist_glob']
time_async = float(data['time_async'])
time_glob = float(data['time_glob'])
RADIUS = float(data['radius'])
K_SPRING = float(data['k_spring'])
GRID_DIVS = int(data['grid_divs'])
N_ATOMS = int(data['n_atoms'])

if 'accumulated_dt' in data:
    accumulated_dt = data['accumulated_dt']
else:
    print("Warning: 'accumulated_dt' not found in NPZ. Using snapshot as placeholder.")
    accumulated_dt = dt_grid_snap * len(e_hist_async) * 10 

print(f"Data Loaded! N={N_ATOMS}. Global Time: {time_glob:.1f}s | Async Time: {time_async:.1f}s")

# ==========================================
# 3. RECONSTRUCT PHYSICS LOCALLY (Force Chains & g(r))
# ==========================================
print("Reconstructing Physical Force Chains...")
cutoff = 2.0 * RADIUS
lines = []
force_mags = []
distances = []  

for i in range(N_ATOMS):
    for j in range(i+1, N_ATOMS):
        dx_raw = pos_async[i,0] - pos_async[j,0]
        dy_raw = pos_async[i,1] - pos_async[j,1]
        dx = dx_raw - np.round(dx_raw)
        dy = dy_raw - np.round(dy_raw)
        dist = np.sqrt(dx**2 + dy**2)
        distances.append(dist)
        
        if dist < cutoff:
            lines.append([(pos_async[i,0], pos_async[i,1]), (pos_async[i,0]-dx, pos_async[i,1]-dy)])
            force_mags.append(K_SPRING * (cutoff - dist))

force_array = np.array(force_mags)
distances = np.array(distances)

# ==========================================
# 4. RENDER FIGURE 1: Phase Space Mapping
# ==========================================
print("Rendering Figure 1...")
fig1, ax1 = plt.subplots(figsize=(8, 7))

dt_grid = dt_grid_snap.reshape(GRID_DIVS, GRID_DIVS).T
heatmap1 = ax1.imshow(dt_grid, cmap='Blues_r', origin='lower', extent=[0, 1, 0, 1], alpha=0.7)
ax1.scatter(pos_async[:, 0], pos_async[:, 1], s=0.5, c='black', alpha=0.2)

if len(lines) > 0:
    lw = (force_array / np.max(force_array)) * 2.5 + 0.2
    lc = LineCollection(lines, cmap='autumn', linewidths=lw)
    lc.set_array(force_array)
    ax1.add_collection(lc)

ax1.set_xticks([]); ax1.set_yticks([])
fig1.colorbar(heatmap1, ax=ax1, fraction=0.046, pad=0.04, label="Local Time Step (dt)")
plt.savefig('Fig1_Bifurcation.pdf', bbox_inches='tight')

# ==========================================
# 5. RENDER FIGURE 2: Dynamical Bifurcation
# ==========================================
print("Rendering Figure 2...")
fig2, (ax2a, ax2b) = plt.subplots(1, 2, figsize=(12, 5))

ax2a.hist(dt_grid_snap, bins=20, color='rebeccapurple', edgecolor='black')
ax2a.set_yscale('log')
ax2a.set_xlabel("Local Domain Time Step")
ax2a.set_ylabel("Domain Count")
ax2a.grid(alpha=0.3)

steps = np.arange(len(dt_hist_glob)) * 10
ax2b.plot(steps, dt_hist_glob, 'k--', linewidth=1.5, label='Global FIRE (Braking)')
ax2b.plot(steps, dt_hist_async, 'r-', linewidth=2, label='Async FIRE (Averaged)')
ax2b.set_xlabel("Algorithmic Step")
ax2b.set_ylabel("Time Step Size")
ax2b.legend()
ax2b.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('Fig2_Dynamics.pdf', bbox_inches='tight')

# ==========================================
# 6. RENDER FIGURE 3: Topological Invariance
# ==========================================
print("Rendering Figure 3...")
fig3, ax3 = plt.subplots(figsize=(8, 6))

steps_e = np.arange(len(e_hist_glob)) * 10
ax3.plot(steps_e, e_hist_glob, 'k--', linewidth=2, label='Global FIRE')
ax3.plot(steps_e, e_hist_async, 'r-', linewidth=2, label='Async FIRE')
ax3.set_yscale('log')
ax3.set_xlabel("Integration Steps")
ax3.set_ylabel("Total Potential Energy")
ax3.legend(loc='lower left')
ax3.grid(alpha=0.3)

axins = inset_locator.inset_axes(ax3, width="40%", height="40%", loc='upper right')
dr = 0.002
r_max = 5 * RADIUS
bins = np.arange(0, r_max, dr)
hist, _ = np.histogram(distances, bins=bins)
r_centers = (bins[1:] + bins[:-1]) / 2.0
rho = N_ATOMS / (1.0**2)
ideal = 2 * np.pi * r_centers * dr * rho * N_ATOMS
g_r = hist / ideal

axins.plot(r_centers / (2*RADIUS), g_r, 'b-', linewidth=1.5)
axins.axvline(1.0, color='r', linestyle='--')
axins.set_xlim(0.8, 2.0)
axins.set_xlabel("$r / 2R$", fontsize=10)
axins.set_ylabel("g(r)", fontsize=10)
axins.set_title("Radial Dist.", fontsize=10)
plt.savefig('Fig3_Energy_GR.pdf', bbox_inches='tight')

# ==========================================
# 7. RENDER FIGURE 4: HPC Wall-Clock Speedup
# ==========================================
print("Rendering Figure 4...")
fig4, ax4 = plt.subplots(figsize=(6, 5))
labels = ['Global FIRE', 'Async FIRE']
times = [time_glob, time_async]
colors = ['black', 'crimson']

bars = ax4.bar(labels, times, color=colors, width=0.5)
ax4.set_ylabel("Wall-Clock Time to Convergence (s)")
ax4.set_title(f"HPC Node Efficiency (N={N_ATOMS})")

speedup = time_glob / time_async
ax4.text(1, time_async + (time_glob*0.05), f"{speedup:.1f}x Speedup", ha='center', va='bottom', fontweight='bold', color='crimson')

plt.savefig('Fig4_Speedup.pdf', bbox_inches='tight')

# ==========================================
# 8. RENDER FIGURE 5: Accumulated "Virtual Age"
# ==========================================
print("Rendering Figure 5...")
fig5, ax5 = plt.subplots(figsize=(8, 7))

age_grid = accumulated_dt.reshape(GRID_DIVS, GRID_DIVS).T
heatmap5 = ax5.imshow(age_grid, cmap='magma', origin='lower', extent=[0, 1, 0, 1], alpha=0.85)

if len(lines) > 0:
    lw = (force_array / np.max(force_array)) * 1.5 + 0.1
    lc5 = LineCollection(lines, colors='white', linewidths=lw, alpha=0.5)
    ax5.add_collection(lc5)

ax5.set_xticks([]); ax5.set_yticks([])
fig5.colorbar(heatmap5, ax=ax5, fraction=0.046, pad=0.04, label=r"Total Accumulated Virtual Time ($\sum \Delta t$)")
ax5.set_title("Phase Space 'Virtual Age' Discrepancy\n(Dark = Frozen Force Chains | Bright = Evolved Rattlers)", pad=15)

plt.savefig('Fig5_Virtual_Age.pdf', bbox_inches='tight')

# ==========================================
# 9. RENDER FIGURE 6: Energy vs. Wall-Clock Time
# ==========================================
print("Rendering Figure 6...")
fig6, ax6 = plt.subplots(figsize=(8, 6))

# Map the steps linearly across the total measured wall-clock time
time_arr_glob = np.linspace(0, time_glob, len(e_hist_glob))
time_arr_async = np.linspace(0, time_async, len(e_hist_async))

ax6.plot(time_arr_glob, e_hist_glob, 'k--', linewidth=2, label='Global FIRE')
ax6.plot(time_arr_async, e_hist_async, 'r-', linewidth=2, label='Async FIRE')
ax6.set_yscale('log')
ax6.set_xlabel("Estimated Wall-Clock Time (s)")
ax6.set_ylabel("Total Potential Energy")
ax6.set_title("Convergence vs. True Computational Time")
ax6.legend(loc='lower left')
ax6.grid(alpha=0.3)

plt.savefig('Fig6_Energy_vs_Time.pdf', bbox_inches='tight')

print("\nAll 6 PRL Figures generated and saved as PDFs!")
plt.show()